In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

FILE_PATH = (
    r"C:\Users\ThinkPad\Documents\Data Science Projekt\RQ 4"
    r"\berlin_pigeon_pullution_2020_2024.csv"
)


def load_and_prepare_data(path):
    # Load data and prepare smoothing
    df = pd.read_csv(path, index_col=0)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")
    
    # Filter/Change data (only positive values and extract the year as string)
    df = df[df["O3"] >= 0].copy()
    df["Year"] = df["Date"].dt.year.astype(str)

    # our smoothing time intervals
    for col, prefix in [("PigeonCount", "P"), ("O3", "O")]:
        df[f"{prefix}_1d"] = df[col]
        df[f"{prefix}_7d"] = df[col].rolling(window=7, center=True).mean()
        df[f"{prefix}_30d"] = df[col].rolling(window=30, center=True).mean()
    
    return df


def create_interactive_plot(df):
    # Creating our interactive plot with a smoothing slider
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # our 3 modes that we can slide trough
    modes = [
        {"name": "Daily", "p_col": "P_1d", "o_col": "O_1d", "visible": False},
        {"name": "Weekly", "p_col": "P_7d", "o_col": "O_7d", "visible": True},
        {"name": "Monthly", "p_col": "P_30d", "o_col": "O_30d", "visible": False}
    ]

    for mode in modes:
        # pigeon line
        fig.add_trace(
            go.Scatter(
                x=df["Date"], y=df[mode["p_col"]],
                name=f"Pigeons ({mode['name']})",
                line=dict(color="royalblue", width=2),
                visible=mode["visible"]
            ),
            secondary_y=False
        )
        # ozon line
        fig.add_trace(
            go.Scatter(
                x=df["Date"], y=df[mode["o_col"]],
                name=f"O3 ({mode['name']})",
                line=dict(color="firebrick", dash="dot", width=1.5),
                visible=mode["visible"]
            ),
            secondary_y=True
        )

    # making our slider
    sliders = [dict(
        active=1,
        currentvalue={"prefix": "Glättung: "},
        pad={"t": 50},
        steps=[
            dict(label="Daily", method="update", 
                 args=[{"visible": [True, True, False, False, False, False]}]),
            dict(label="Weekly", method="update", 
                 args=[{"visible": [False, False, True, True, False, False]}]),
            dict(label="Monthly", method="update", 
                 args=[{"visible": [False, False, False, False, True, True]}])
        ]
    )]

    # putting everything in the layout 
    fig.update_layout(
        sliders=sliders,
        title="<b>Pigeon count vs. O3 in 2021-2025</b>",
        template="plotly_white",
        hovermode="x unified"
    )

    fig.update_yaxes(title_text="Pigeon Count", secondary_y=False)
    fig.update_yaxes(title_text="O3 Concentration (µg/m³)", secondary_y=True)
    
    return fig


def create_regression_plot(df):
    # Creating our second plot, the regression line
    fig = px.scatter(
        df,
        x="O3",
        y="PigeonCount",
        color="Year",
        trendline="ols",
        trendline_scope="overall",
        trendline_color_override="black",
        title="<b>Regression Analysis: O3 vs. Pigeon Count</b>",
        labels={"O3": "O3 Concentration (µg/m³)", "PigeonCount": "Pigeon Count"},
        opacity=0.4,
        template="plotly_white",
    )

    fig.update_xaxes(range=[0, df["O3"].max() * 1.05])
    fig.update_yaxes(range=[0, 130])
    
    return fig

def create_binned_histogram(df):
    # Creating a Histogram that bins the o3 values into 10 bins
    # and shows how many pigeons in total were counted at those values
    
    # Creating the bins
    bin_edges = list(range(0, 111, 10))
    df_binned = df.copy()
    df_binned["O3_Bin"] = pd.cut(df_binned["O3"], bins=bin_edges)
    
    # Group and aggregate the sum
    df_plot = df_binned.groupby("O3_Bin", observed=True).agg({
        "PigeonCount": "sum"
    }).reset_index()

    df_plot["O3_Range"] = df_plot["O3_Bin"].astype(str)

    # Creating the plot itself
    fig = px.bar(
        df_plot,
        x="O3_Range",
        y="PigeonCount",
        title="<b>Distribution: Average Pigeon Count by O3 Bins</b>",
        labels={
            "O3": "O3 Concentration (µg/m³)",
            "PigeonCount": "Total Pigeon Count"
        },
        template="plotly_white",
        color_discrete_sequence=["royalblue"]
    )

    # Creating the layout or rather the axises
    fig.update_layout(
        bargap=0.1,
        xaxis={
            'title': 'O3 Concentration (µg/m³)',
            'categoryorder': 'array', 
            'categoryarray': df_plot["O3_Range"]
        },
        yaxis_title="Total Pigeon Count"
    )

    return fig


# Executing the functions
if __name__ == "__main__":
    pigeon_data = load_and_prepare_data(FILE_PATH)
    
    time_series_fig = create_interactive_plot(pigeon_data)
    time_series_fig.show()
    
    regression_fig = create_regression_plot(pigeon_data)
    regression_fig.show()

    hist_fig = create_binned_histogram(pigeon_data)
    hist_fig.show()